# Linear Programming with Python & AMPL

**Graduate Optimization Course** | Northern Illinois University

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Write AMPL model files (`.mod`) that define decision variables, objectives, and constraints
2. Create AMPL data files (`.dat`) to separate problem data from model logic
3. Use Python to orchestrate the modeling workflow
4. Interpret optimal solutions, shadow prices, and reduced costs
5. Apply these skills to classic LP problems: resource allocation, blending, and staffing

---

## Why Separate Model, Data, and Code?

| Component | File Type | Purpose |
|:----------|:----------|:--------|
| **Model** | `.mod` | Mathematical structure (variables, objective, constraints) |
| **Data** | `.dat` | Problem-specific numbers (costs, capacities, demands) |
| **Code** | `.py` / notebook | Orchestration, analysis, visualization |

**Benefits:**
- Reuse the same model with different data instances
- Syntax highlighting and debugging in text editors
- Version control friendly
- Industry standard practice

---

## Part 1: AMPL Syntax Reference

Read this section first to understand AMPL syntax before diving into examples.

### 1.1 Sets

Sets define the indices used in parameters, variables, and constraints.

```ampl
set PRODUCTS;                         # Simple set (members defined in .dat file)
set MONTHS ordered;                   # Ordered set (enables prev(), next(), first(), last())
set ARCS within {NODES, NODES};       # Set of pairs (subset of cross product)
set VALID_COMBOS := {(i,j) in ARCS: cost[i,j] < 100};  # Conditional set
```

### 1.2 Parameters

Parameters are known constants (data inputs).

```ampl
param demand {PRODUCTS};              # Indexed by one set
param cost {PRODUCTS, MONTHS};        # Indexed by two sets
param discount default 0.1;           # Scalar with default value
param big_M := 1000000;               # Defined inline (no data file needed)
param num_products := card(PRODUCTS); # Computed from set cardinality
```

### 1.3 Decision Variables

Variables are what the solver determines.

```ampl
var x {PRODUCTS} >= 0;                       # Non-negative continuous
var y {PRODUCTS} >= 0 integer;               # Non-negative integer
var z {PRODUCTS} binary;                     # Binary (0 or 1)
var w {j in PRODUCTS} >= lb[j], <= ub[j];    # Variable bounds from parameters
var flow {ARCS} >= 0, <= capacity;           # Indexed with upper bound
```

### 1.4 Objective Function

Exactly one objective: either `maximize` or `minimize`.

```ampl
maximize Total_Profit:
    sum {j in PRODUCTS} price[j] * x[j];

minimize Total_Cost:
    sum {i in PLANTS, j in CUSTOMERS} cost[i,j] * ship[i,j];
```

### 1.5 Constraints

Use `subject to` (or `s.t.`) to define constraints.

```ampl
# Single constraint
subject to Budget:
    sum {j in PRODUCTS} cost[j] * x[j] <= budget;

# Family of constraints (one for each element in set)
subject to Capacity {i in PLANTS}:
    sum {j in PRODUCTS} x[i,j] <= capacity[i];

# Conditional constraint (only when condition is true)
subject to Min_Order {j in PRODUCTS: min_qty[j] > 0}:
    x[j] >= min_qty[j];

# Inventory balance (using ordered sets)
subject to Balance {t in MONTHS: ord(t) > 1}:
    inventory[t] = inventory[prev(t)] + production[t] - demand[t];
```

### 1.6 Data File Syntax (.dat)

Data files provide values for sets and parameters.

```ampl
# Define set members
set PRODUCTS := TableA TableB ChairX;
set MONTHS := Jan Feb Mar Apr;

# Simple scalar
param budget := 50000;

# 1-D parameter (list format)
param demand :=
    TableA  100
    TableB  200
    ChairX  150;

# 2-D parameter (table format)
param cost:        Jan   Feb   Mar   Apr :=
    TableA          10    12    11    13
    TableB          15    14    16    15
    ChairX          20    18    19    21;

# 2-D parameter (list format - alternative)
param distance :=
    [NYC, Boston]     215
    [NYC, Philly]      95
    [Boston, Philly]  310;
```

### 1.7 Useful Built-in Functions

| Function | Description | Example |
|:---------|:------------|:--------|
| `sum {i in S}` | Summation over set | `sum {j in PRODUCTS} x[j]` |
| `card(S)` | Number of elements in set | `param n := card(PRODUCTS);` |
| `ord(i)` | Position of element in ordered set (1-based) | `ord(t) > 1` |
| `first(S)` | First element of ordered set | `first(MONTHS)` |
| `last(S)` | Last element of ordered set | `last(MONTHS)` |
| `prev(i)` | Previous element in ordered set | `inventory[prev(t)]` |
| `next(i)` | Next element in ordered set | `demand[next(t)]` |
| `member(k, S)` | k-th element of ordered set | `member(3, MONTHS)` |

---

## Part 2: Environment Setup

We use `ampl_notebook()` which is designed specifically for cloud notebook environments like Google Colab.

In [ ]:
# Install AMPL Python API
%pip install -q amplpy

In [ ]:
from amplpy import AMPL, ampl_notebook

# Initialize AMPL with solvers for Colab environment
# Get your own free license at: https://ampl.com/ce or https://ampl.com/courses
ampl_notebook(
    modules=["highs"],  # Open-source solver; add "gurobi", "cplex" if needed
    license_uuid="YOUR-AMPL-LICENSE-UUID"
)

print("✓ AMPL environment ready!")

---

## Part 3: Resource Allocation Problem

### Problem Description

A furniture company produces **tables** and **chairs**. Each product requires labor and wood.

| Product | Profit (\$/unit) | Labor (hrs/unit) | Wood (board-ft/unit) |
|:--------|:-----------------|:-----------------|:---------------------|
| Table   | 70               | 4                | 30                   |
| Chair   | 50               | 3                | 20                   |

**Available resources:**
- Labor: 240 hours
- Wood: 1,200 board-feet

**Goal:** Maximize total profit.

### Mathematical Formulation

**Sets:**
- $J$ = set of products
- $I$ = set of resources

**Parameters:**
- $p_j$ = profit per unit of product $j$
- $a_{ij}$ = amount of resource $i$ needed per unit of product $j$
- $b_i$ = available amount of resource $i$

**Decision Variables:**
- $x_j$ = number of units of product $j$ to produce

**Formulation:**

$$\max \sum_{j \in J} p_j \cdot x_j$$

$$\text{s.t.} \quad \sum_{j \in J} a_{ij} \cdot x_j \leq b_i \quad \forall i \in I$$

$$x_j \geq 0 \quad \forall j \in J$$

### Step 1: Create the Model File

The `%%writefile` magic command writes the cell contents directly to a file.

In [ ]:
%%writefile resource.mod
# ===========================================
# Resource Allocation Model
# ===========================================

# --- SETS ---
set PRODUCTS;                          # Set of products to produce
set RESOURCES;                         # Set of limited resources

# --- PARAMETERS ---
param profit {PRODUCTS};               # Profit per unit of each product
param usage {RESOURCES, PRODUCTS};     # Resource usage per unit of product
param available {RESOURCES};           # Available amount of each resource

# --- DECISION VARIABLES ---
var Produce {j in PRODUCTS} >= 0;      # Units of product j to produce

# --- OBJECTIVE FUNCTION ---
maximize Total_Profit:
    sum {j in PRODUCTS} profit[j] * Produce[j];

# --- CONSTRAINTS ---
subject to Resource_Limit {i in RESOURCES}:
    sum {j in PRODUCTS} usage[i,j] * Produce[j] <= available[i];

### Step 2: Create the Data File

In [ ]:
%%writefile resource.dat
# ===========================================
# Data for Furniture Production Problem
# ===========================================

# Define the sets
set PRODUCTS := Tables Chairs;
set RESOURCES := Labor Wood;

# Profit per unit ($)
param profit :=
    Tables  70
    Chairs  50;

# Resource usage: usage[resource, product]
param usage:        Tables  Chairs :=
    Labor           4       3
    Wood            30      20;

# Available resources
param available :=
    Labor   240
    Wood    1200;

### Step 3: Solve with Python

In [ ]:
# Create model instance for the resource allocation problem
furniture_model = AMPL()

# Read model and data files
furniture_model.read('resource.mod')
furniture_model.read_data('resource.dat')

# Set solver and solve
furniture_model.option['solver'] = 'highs'
furniture_model.solve()

# Check solution status
solve_result = furniture_model.get_value('solve_result')
print(f"Solver Status: {solve_result}")

### Step 4: Analyze Results

In [ ]:
# Get optimal objective value
total_profit = furniture_model.obj['Total_Profit'].value()
print(f"Maximum Profit: ${total_profit:,.2f}")
print()

# Get optimal production quantities
print("Optimal Production Plan:")
produce = furniture_model.var['Produce']
for product in furniture_model.set['PRODUCTS'].members():
    qty = produce[product].value()
    print(f"  {product}: {qty:.1f} units")

In [ ]:
# Analyze resource utilization and shadow prices
print("Resource Analysis:")
print("=" * 55)

resource_limit = furniture_model.con['Resource_Limit']
available_param = furniture_model.param['available']

for resource in furniture_model.set['RESOURCES'].members():
    used = resource_limit[resource].body()
    avail = available_param[resource]
    slack = avail - used
    shadow = resource_limit[resource].dual()

    print(f"\n{resource}:")
    print(f"  Used: {used:.1f} / {avail:.1f} (Slack: {slack:.1f})")
    print(f"  Shadow Price: ${shadow:.2f}")

    if shadow > 0.001:
        print(f"  → Binding! Each additional unit worth ${shadow:.2f}")
    else:
        print(f"  → Non-binding (slack resources available)")

### Understanding Shadow Prices

The **shadow price** (dual value) of a constraint tells us:
- How much the objective improves if we increase the RHS by 1 unit
- The maximum we should pay for one additional unit of that resource

**Example:** If the shadow price of Labor is \$10, then:
- Adding 1 more labor hour would increase profit by \$10
- We should pay up to \$10/hour for overtime

---

## Part 4: Blending Problem

### Problem Description

A feed company must blend ingredients to create animal feed that meets nutritional requirements at minimum cost.

**Ingredients available:**

| Ingredient | Cost (\$/lb) | Protein (%) | Fiber (%) | Max Available (lbs) |
|:-----------|:-------------|:------------|:----------|:--------------------|
| Corn       | 0.30         | 9           | 2         | 500                 |
| Oats       | 0.25         | 12          | 6         | 400                 |
| Soybeans   | 0.45         | 40          | 5         | 300                 |

**Requirements for 1,000 lbs of feed:**
- Minimum protein: 15%
- Maximum fiber: 5%

### Mathematical Formulation

**Sets:**
- $J$ = set of ingredients
- $N$ = set of nutrients

**Parameters:**
- $c_j$ = cost per lb of ingredient $j$
- $a_{nj}$ = fraction of nutrient $n$ in ingredient $j$
- $\underline{r}_n, \overline{r}_n$ = min/max fraction of nutrient $n$ required
- $u_j$ = max available amount of ingredient $j$
- $T$ = total amount to produce

**Decision Variables:**
- $x_j$ = pounds of ingredient $j$ to use

**Formulation:**

$$\min \sum_{j \in J} c_j \cdot x_j$$

$$\text{s.t.} \quad \sum_{j \in J} x_j = T$$

$$\sum_{j \in J} a_{nj} \cdot x_j \geq \underline{r}_n \cdot T \quad \forall n \in N$$

$$\sum_{j \in J} a_{nj} \cdot x_j \leq \overline{r}_n \cdot T \quad \forall n \in N$$

$$0 \leq x_j \leq u_j \quad \forall j \in J$$

In [ ]:
%%writefile blending.mod
# ===========================================
# Blending Problem Model
# ===========================================

# --- SETS ---
set INGREDIENTS;                       # Available ingredients
set NUTRIENTS;                         # Nutrients to track

# --- PARAMETERS ---
param cost {INGREDIENTS};              # Cost per unit of ingredient
param nutrient_content {NUTRIENTS, INGREDIENTS};  # Nutrient fraction in each ingredient
param min_nutrient {NUTRIENTS} default 0;         # Min nutrient requirement (fraction)
param max_nutrient {NUTRIENTS} default 1;         # Max nutrient allowed (fraction)
param available {INGREDIENTS};         # Maximum available of each ingredient
param total_amount;                    # Total amount to produce

# --- DECISION VARIABLES ---
var Use {j in INGREDIENTS} >= 0, <= available[j];

# --- OBJECTIVE ---
minimize Total_Cost:
    sum {j in INGREDIENTS} cost[j] * Use[j];

# --- CONSTRAINTS ---
subject to Total_Weight:
    sum {j in INGREDIENTS} Use[j] = total_amount;

subject to Min_Nutrient {n in NUTRIENTS: min_nutrient[n] > 0}:
    sum {j in INGREDIENTS} nutrient_content[n,j] * Use[j] >= min_nutrient[n] * total_amount;

subject to Max_Nutrient {n in NUTRIENTS: max_nutrient[n] < 1}:
    sum {j in INGREDIENTS} nutrient_content[n,j] * Use[j] <= max_nutrient[n] * total_amount;

In [ ]:
%%writefile blending.dat
# ===========================================
# Data for Animal Feed Blending Problem
# ===========================================

set INGREDIENTS := Corn Oats Soybeans;
set NUTRIENTS := Protein Fiber;

# Total pounds of feed to produce
param total_amount := 1000;

# Cost per pound
param cost :=
    Corn      0.30
    Oats      0.25
    Soybeans  0.45;

# Maximum available (pounds)
param available :=
    Corn      500
    Oats      400
    Soybeans  300;

# Nutrient content (as decimal, e.g., 0.09 = 9%)
param nutrient_content:   Corn    Oats    Soybeans :=
    Protein               0.09    0.12    0.40
    Fiber                 0.02    0.06    0.05;

# Minimum nutrient requirements (as fraction)
param min_nutrient :=
    Protein  0.15;

# Maximum nutrient allowed (as fraction)
param max_nutrient :=
    Fiber    0.05;

In [ ]:
# Create model instance for the blending problem
blend_model = AMPL()
blend_model.read('blending.mod')
blend_model.read_data('blending.dat')
blend_model.option['solver'] = 'highs'
blend_model.solve()

print(f"Solver Status: {blend_model.get_value('solve_result')}")
print(f"\nMinimum Cost: ${blend_model.obj['Total_Cost'].value():,.2f}")
print(f"Cost per pound: ${blend_model.obj['Total_Cost'].value()/1000:.4f}")

In [ ]:
# Display optimal blend
print("Optimal Blend:")
print("=" * 40)

use = blend_model.var['Use']
total = blend_model.param['total_amount'].value()

for ing in blend_model.set['INGREDIENTS'].members():
    amount = use[ing].value()
    pct = amount / total * 100
    print(f"  {ing:12s}: {amount:7.1f} lbs  ({pct:5.1f}%)")

# Verify nutrient content
print("\nNutrient Verification:")
print("=" * 40)

content = blend_model.param['nutrient_content']
for nutrient in blend_model.set['NUTRIENTS'].members():
    total_nutrient = sum(
        content[nutrient, ing] * use[ing].value()
        for ing in blend_model.set['INGREDIENTS'].members()
    )
    pct = total_nutrient / total * 100
    print(f"  {nutrient:12s}: {pct:.2f}%")

---

## Part 5: Staff Scheduling Problem

### Problem Description

A call center operates 7 days a week. Each employee works **5 consecutive days**, then has 2 days off. The center must meet minimum staffing requirements each day at minimum cost.

| Day       | Mon | Tue | Wed | Thu | Fri | Sat | Sun |
|:----------|:----|:----|:----|:----|:----|:----|:----|
| Min Staff | 17  | 13  | 15  | 19  | 14  | 16  | 11  |
| Cost/Day  | 180 | 180 | 180 | 180 | 180 | 240 | 240 |

Weekend days (Sat, Sun) have premium pay.

### Mathematical Formulation

**Key Insight:** An employee starting on day $d$ works days $d, d+1, d+2, d+3, d+4$ (mod 7).

**Decision Variables:**
- $x_d$ = number of employees starting their 5-day shift on day $d$

**Objective:** Minimize total weekly labor cost

$$\min \sum_{d} x_d \cdot \left( \sum_{k=0}^{4} c_{(d+k) \mod 7} \right)$$

**Constraints:** Each day $d$ must have enough staff from employees who started on days $d, d-1, d-2, d-3, d-4$.

$$\sum_{k=0}^{4} x_{(d-k) \mod 7} \geq r_d \quad \forall d$$

In [ ]:
%%writefile staffing.mod
# ===========================================
# Staff Scheduling Model
# Employees work 5 consecutive days, then 2 days off
# ===========================================

# --- SETS ---
set DAYS ordered;                      # Days of the week (ordered for cycling)

# --- PARAMETERS ---
param min_staff {DAYS};                # Minimum staff required each day
param daily_cost {DAYS};               # Cost per employee per day worked
param num_days := card(DAYS);          # Number of days in cycle (7)
param shift_length := 5;               # Consecutive days per shift

# --- DECISION VARIABLES ---
var Start {d in DAYS} >= 0 integer;    # Employees starting on day d

# --- OBJECTIVE ---
# Minimize total weekly cost: each employee is paid for 5 days worked
minimize Total_Cost:
    sum {d in DAYS, k in 0..shift_length-1}
        daily_cost[member(((ord(d) - 1 + k) mod num_days) + 1, DAYS)] * Start[d];

# --- CONSTRAINTS ---
# Day d is covered by employees starting on d, d-1, d-2, d-3, d-4
subject to Staff_Requirement {d in DAYS}:
    sum {k in 0..shift_length-1}
        Start[member(((ord(d) - 1 - k + num_days) mod num_days) + 1, DAYS)]
    >= min_staff[d];

In [ ]:
%%writefile staffing.dat
# ===========================================
# Data for Call Center Staffing Problem
# ===========================================

set DAYS := Mon Tue Wed Thu Fri Sat Sun;

# Minimum staff required each day
param min_staff :=
    Mon  17
    Tue  13
    Wed  15
    Thu  19
    Fri  14
    Sat  16
    Sun  11;

# Cost per employee per day worked (weekend premium)
param daily_cost :=
    Mon  180
    Tue  180
    Wed  180
    Thu  180
    Fri  180
    Sat  240
    Sun  240;

In [ ]:
# Create model instance for the staffing problem
staffing_model = AMPL()
staffing_model.read('staffing.mod')
staffing_model.read_data('staffing.dat')
staffing_model.option['solver'] = 'highs'
staffing_model.solve()

print(f"Solver Status: {staffing_model.get_value('solve_result')}")
print(f"\nMinimum Weekly Cost: ${staffing_model.obj['Total_Cost'].value():,.2f}")

In [ ]:
# Display staffing schedule
print("Optimal Staffing Schedule:")
print("=" * 55)

start = staffing_model.var['Start']
days = list(staffing_model.set['DAYS'].members())

print("\nEmployees Starting Each Day:")
total_employees = 0
for day in days:
    num = int(start[day].value())
    total_employees += num
    work_days = [days[(days.index(day) + k) % 7] for k in range(5)]
    print(f"  {day}: {num:2d} employees  →  work {', '.join(work_days)}")

print(f"\nTotal Employees Needed: {total_employees}")

In [ ]:
# Verify staffing levels meet requirements
print("Daily Staffing Verification:")
print("=" * 50)
print(f"{'Day':<6}  {'Required':<10}  {'Scheduled':<10}  {'Status'}")
print("-" * 50)

min_staff = staffing_model.param['min_staff']

for i, day in enumerate(days):
    required = int(min_staff[day])
    # Count employees working on this day (started in last 5 days)
    scheduled = sum(
        int(start[days[(i - k) % 7]].value())
        for k in range(5)
    )
    status = "✓" if scheduled >= required else "✗ SHORTAGE"
    print(f"{day:<6}  {required:<10}  {scheduled:<10}  {status}")

---

## Part 6: Exercises

Complete the following exercises to practice your AMPL modeling skills.

---

### Exercise 1: Expanded Resource Allocation

Extend the furniture production problem with additional products and a new resource.

**New data:**

| Product   | Profit (\$) | Labor (hrs) | Wood (bd-ft) | Machine (hrs) |
|:----------|:------------|:------------|:-------------|:--------------|
| Table     | 70          | 4           | 30           | 2.0           |
| Chair     | 50          | 3           | 20           | 1.5           |
| Desk      | 90          | 5           | 40           | 3.0           |
| Bookcase  | 80          | 4           | 35           | 2.5           |

**Available resources:**

| Resource | Available |
|:---------|:----------|
| Labor    | 400 hours |
| Wood     | 2,000 board-feet |
| Machine  | 250 hours |

**Tasks:**
1. Create a new `.dat` file with these values (reuse `resource.mod`)
2. Solve and report the optimal production plan
3. Which resources are binding? What are their shadow prices?

In [ ]:
%%writefile exercise1.dat
# TODO: Complete the data file for Exercise 1

set PRODUCTS := Tables Chairs Desks Bookcases;
set RESOURCES := Labor Wood Machine;

param profit :=
    # TODO: Fill in profit values
    ;

param usage:    Tables  Chairs  Desks  Bookcases :=
    # TODO: Fill in usage table (3 resources x 4 products)
    ;

param available :=
    # TODO: Fill in available resources
    ;

In [ ]:
# Solve Exercise 1 (uncomment after completing the data file)

# exercise1_model = AMPL()
# exercise1_model.read('resource.mod')
# exercise1_model.read_data('exercise1.dat')
# exercise1_model.option['solver'] = 'highs'
# exercise1_model.solve()
#
# print(f"Maximum Profit: ${exercise1_model.obj['Total_Profit'].value():,.2f}")

---

### Exercise 2: Diet Problem

Create a minimum-cost daily meal plan using these foods:

| Food     | Cost (\$) | Calories | Protein (g) | Fat (g) | Carbs (g) | Max Servings |
|:---------|:----------|:---------|:------------|:--------|:----------|:-------------|
| Oatmeal  | 0.50      | 150      | 5           | 2       | 27        | 4            |
| Chicken  | 2.50      | 250      | 30          | 8       | 0         | 3            |
| Rice     | 0.30      | 200      | 4           | 0.5     | 45        | 5            |
| Broccoli | 1.00      | 50       | 4           | 0.5     | 10        | 4            |
| Milk     | 0.80      | 120      | 8           | 5       | 12        | 3            |

**Daily requirements:**

| Nutrient | Minimum | Maximum |
|:---------|:--------|:--------|
| Calories | 2,000   | 2,500   |
| Protein  | 60 g    | —       |
| Fat      | —       | 65 g    |
| Carbs    | 250 g   | 350 g   |

**Tasks:**
1. Write a new model file (`diet.mod`)
2. Write the data file (`diet.dat`)
3. Find the minimum cost diet

**Hint:** This is similar to blending, but without a fixed total amount. Each "nutrient" has its own min/max bounds.

In [ ]:
%%writefile diet.mod
# TODO: Write the Diet Problem model

# SETS
set FOODS;
set NUTRIENTS;

# PARAMETERS
# TODO: Define parameters for cost, nutrient content, limits, max servings

# VARIABLES
# TODO: var Eat {f in FOODS} >= 0, <= max_servings[f];

# OBJECTIVE
# TODO: minimize Total_Cost

# CONSTRAINTS
# TODO: Min and Max nutrient constraints

In [ ]:
%%writefile diet.dat
# TODO: Write the Diet Problem data

---

### Exercise 3: Transportation Problem

A company has 3 warehouses and 4 customers. Minimize shipping cost while meeting all demand.

**Supply at each warehouse:**

| Warehouse | Supply (units) |
|:----------|:---------------|
| W1        | 400            |
| W2        | 300            |
| W3        | 500            |

**Demand at each customer:**

| Customer | Demand (units) |
|:---------|:---------------|
| A        | 250            |
| B        | 350            |
| C        | 300            |
| D        | 200            |

**Shipping cost per unit (\$):**

| From \ To | A  | B  | C  | D  |
|:----------|:---|:---|:---|:---|
| W1        | 8  | 6  | 10 | 9  |
| W2        | 9  | 12 | 13 | 7  |
| W3        | 14 | 9  | 16 | 5  |

**Tasks:**
1. Write `transport.mod`
2. Write `transport.dat`
3. Solve and display the optimal shipping plan as a table

In [ ]:
%%writefile transport.mod
# TODO: Write the Transportation Problem model

# SETS
set WAREHOUSES;
set CUSTOMERS;

# PARAMETERS
# TODO: supply, demand, cost

# VARIABLES
# TODO: var Ship {WAREHOUSES, CUSTOMERS} >= 0;

# OBJECTIVE
# TODO: minimize Total_Cost

# CONSTRAINTS
# TODO: Supply limits, demand satisfaction

In [ ]:
%%writefile transport.dat
# TODO: Write the Transportation Problem data

---

### Exercise 4: Multi-Period Production Planning

A factory must plan production over 4 months. They can hold inventory (at a cost) to meet future demand.

| Month | Demand | Prod. Cost (\$/unit) | Max Production | Holding Cost (\$/unit) |
|:------|:-------|:---------------------|:---------------|:-----------------------|
| Jan   | 100    | 20                   | 120            | 2                      |
| Feb   | 150    | 22                   | 120            | 2                      |
| Mar   | 130    | 21                   | 120            | 2                      |
| Apr   | 180    | 23                   | 120            | 2                      |

**Additional constraints:**
- Starting inventory: 20 units
- Ending inventory (after April): at least 10 units

**Key relationship (inventory balance):**

$$I_t = I_{t-1} + P_t - D_t$$

Where:
- $I_t$ = inventory at end of month $t$
- $P_t$ = production in month $t$
- $D_t$ = demand in month $t$

**Tasks:**
1. Write `production.mod` using an **ordered set** for months
2. Write `production.dat`
3. Find the minimum cost production plan

In [ ]:
%%writefile production.mod
# TODO: Write the Multi-Period Production Planning model

# Hint: Use "set MONTHS ordered;" to enable prev(), first(), last()

# SETS
set MONTHS ordered;

# PARAMETERS
# TODO: demand, prod_cost, max_prod, hold_cost, init_inventory, final_inventory_min

# VARIABLES
# TODO: Produce, Inventory

# OBJECTIVE
# TODO: minimize production cost + holding cost

# CONSTRAINTS
# TODO: Inventory balance for first month (uses init_inventory)
# TODO: Inventory balance for subsequent months (uses prev())
# TODO: Final inventory requirement

In [ ]:
%%writefile production.dat
# TODO: Write the production planning data

---

## Summary

In this notebook, you learned:

1. **AMPL Syntax**: Sets, parameters, variables, objectives, constraints
2. **File Separation**: `.mod` for logic, `.dat` for data, Python for orchestration
3. **Classic LP Problems**:
   - Resource Allocation (maximize profit subject to resource limits)
   - Blending (minimize cost while meeting specifications)
   - Staff Scheduling (minimize cost while meeting coverage)
4. **Solution Analysis**: Optimal values, shadow prices, binding constraints

### Assignment

Re-implement the **"Ohio National Bank"** model from the textbook using the `.mod` + `.dat` approach demonstrated in this notebook.